# 12 — Full-Universe Ingestion (Milestone M5)

**DSML stage:** scaled data ops. Runs the proven Nvidia pipeline (notebooks 01–09) over the **whole
13-filer universe** plus Federal Register export controls. Every stage is **checkpointed to disk and
idempotent** — interrupt and re-run at any time; completed work is never redone or re-billed.

| Stage | Cost | Checkpoint |
|---|---|---|
| 1. Acquire filings (annuals 2023+, latest quarterly) | free | file exists on disk |
| 2. XBRL metrics (all filers) | free | parquet per filer |
| 3. Parse + chunk (10-K/10-Q/20-F) | free | parquet per filer |
| 4. Deterministic graph load | free | idempotent MERGE |
| 5. LLM extraction *(the paid stage — estimate shown first)* | ~$8–14 | jsonl per filer |
| 6. Resolve + embed + load knowledge | free (local CPU) | parquet + MERGE |
| 7. Export controls (Federal Register → `ExportControl` + `AFFECTED_BY`) | free | idempotent MERGE |

**Form-type reality (from notebook 01's survey):** US filers file 10-K/10-Q; **TSMC and ASML file 20-F**
(annual only — kept items: 3 Key Information/Risk, 4 Business, 5 Operating Review); **Samsung doesn't file**
with the SEC and exists only as a mentioned entity.

**Extraction scope (cost control):** all kept sections of each filer's **latest annual + latest quarterly**,
plus **risk-factor sections only** of historical annuals (2023+) — that history is what powers temporal
versioning in notebook 13.

In [13]:
import json
import os
import re
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import tiktoken
from dotenv import load_dotenv
from neo4j import GraphDatabase

warnings.filterwarnings("ignore", category=UserWarning)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(PROJECT_ROOT / ".env")

import edgar
SEC_USER_AGENT = os.getenv("SEC_USER_AGENT", "Amit Badave amit11badave.ab@gmail.com")
edgar.set_identity(SEC_USER_AGENT)

driver = GraphDatabase.driver(os.environ["NEO4J_URI"], auth=(os.environ["NEO4J_USER"], os.environ["NEO4J_PASSWORD"]))
driver.verify_connectivity()

ENC = tiktoken.get_encoding("cl100k_base")
CANONICAL = json.loads((PROJECT_ROOT / "artifacts/canonical_entities.json").read_text())

# ticker: (canonical name, annual form, quarterly form or None)
FILERS = {
    "NVDA": ("Nvidia", "10-K", "10-Q"), "AMD": ("AMD", "10-K", "10-Q"),
    "INTC": ("Intel", "10-K", "10-Q"), "AVGO": ("Broadcom", "10-K", "10-Q"),
    "QCOM": ("Qualcomm", "10-K", "10-Q"), "MU": ("Micron", "10-K", "10-Q"),
    "AAPL": ("Apple", "10-K", "10-Q"), "MSFT": ("Microsoft", "10-K", "10-Q"),
    "AMZN": ("Amazon", "10-K", "10-Q"), "GOOGL": ("Alphabet", "10-K", "10-Q"),
    "META": ("Meta", "10-K", "10-Q"),
    "TSM": ("TSMC", "20-F", None), "ASML": ("ASML", "20-F", None),
}
ANNUAL_SINCE = 2023  # annuals filed this year or later

RAW_EDGAR = PROJECT_ROOT / "data/raw/edgar"
RAW_XBRL = PROJECT_ROOT / "data/raw/xbrl"
SECTIONS_DIR = PROJECT_ROOT / "data/interim/sections"
SECTION_TEXT_DIR = PROJECT_ROOT / "data/interim/section_texts"
CHUNKS_DIR = PROJECT_ROOT / "data/processed/chunks"
XBRL_OUT = PROJECT_ROOT / "data/processed/xbrl"
EXTRACT_DIR = PROJECT_ROOT / "data/processed/extractions"
EMB_DIR = PROJECT_ROOT / "data/processed/embeddings"
for p in (RAW_EDGAR, RAW_XBRL, SECTIONS_DIR, SECTION_TEXT_DIR, CHUNKS_DIR, XBRL_OUT, EXTRACT_DIR, EMB_DIR):
    p.mkdir(parents=True, exist_ok=True)
print(f"{len(FILERS)} filers | Neo4j connected")

13 filers | Neo4j connected


## Stage 1 — Acquire filings (free; skips files already on disk)

In [2]:
def acquire_company(ticker: str) -> list[dict]:
    """Download annuals (>= ANNUAL_SINCE) + latest quarterly for one filer; return manifest rows."""
    name, annual_form, quarterly_form = FILERS[ticker]
    company = edgar.Company(ticker)
    cdir = RAW_EDGAR / ticker
    cdir.mkdir(exist_ok=True)
    targets = [f for f in company.get_filings(form=annual_form) if f.filing_date.year >= ANNUAL_SINCE]
    if quarterly_form:
        targets.append(company.get_filings(form=quarterly_form).latest(1))
    rows = []
    for f in targets:
        local = cdir / f"{f.form.replace('/', '-')}_{f.filing_date}_{f.accession_no}.html"
        if not local.exists():
            local.write_text(f.html(), encoding="utf-8")
            time.sleep(0.15)  # stay well under SEC's 10 req/s
        rows.append({"ticker": ticker, "cik": f.cik, "form": f.form, "filing_date": str(f.filing_date),
                     "accession_no": f.accession_no, "source_url": f.document.url,
                     "local_path": str(local.relative_to(PROJECT_ROOT)), "size_bytes": local.stat().st_size})
    return rows

manifest_path = RAW_EDGAR / "manifest_universe.json"
manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}
for ticker in FILERS:
    if ticker not in manifest:
        manifest[ticker] = acquire_company(ticker)
        manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
        print(f"{ticker}: {len(manifest[ticker])} filings downloaded")
    else:
        print(f"{ticker}: {len(manifest[ticker])} filings (cached)")
ALL_FILINGS = [row for rows in manifest.values() for row in rows]
print(f"total: {len(ALL_FILINGS)} filings")

NVDA: 5 filings (cached)
AMD: 6 filings (cached)
INTC: 5 filings (cached)
AVGO: 4 filings (cached)
QCOM: 4 filings (cached)
MU: 4 filings (cached)
AAPL: 4 filings (cached)
MSFT: 4 filings (cached)
AMZN: 5 filings (cached)
GOOGL: 5 filings (cached)
META: 5 filings (cached)
TSM: 4 filings (cached)
ASML: 4 filings (cached)
total: 59 filings


## Stage 2 — XBRL metrics for all filers

US filers report under **us-gaap**; TSMC/ASML under **ifrs-full** — each metric maps to a concept list
spanning both taxonomies. Same first-disclosure dedup as notebook 02.

In [3]:
import urllib.request

KEY_CONCEPTS = {
    "revenue": ["Revenues", "RevenueFromContractWithCustomerExcludingAssessedTax", "Revenue"],
    "capex": ["PaymentsToAcquirePropertyPlantAndEquipment", "PurchaseOfPropertyPlantAndEquipmentClassifiedAsInvestingActivities"],
    "rnd": ["ResearchAndDevelopmentExpense", "ResearchAndDevelopmentExpenditure"],
    "net_income": ["NetIncomeLoss", "ProfitLoss"],
}
ANNUAL_FORMS = {"10-K", "20-F"}

def fetch_companyfacts(cik: int) -> dict:
    raw = RAW_XBRL / f"CIK{cik:010d}_companyfacts.json"
    if not raw.exists():
        req = urllib.request.Request(f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik:010d}.json",
                                     headers={"User-Agent": SEC_USER_AGENT})
        raw.write_bytes(urllib.request.urlopen(req).read())
        time.sleep(0.15)
    return json.loads(raw.read_text())

def curate_metrics(ticker: str) -> pd.DataFrame:
    cik = manifest[ticker][0]["cik"]
    facts_json = fetch_companyfacts(cik)
    rows = []
    for taxonomy in ("us-gaap", "ifrs-full"):
        for concept, payload in facts_json["facts"].get(taxonomy, {}).items():
            for unit, facts in payload["units"].items():
                for fact in facts:
                    rows.append({"concept": concept, "unit": unit, **{k: fact.get(k) for k in
                                 ("start", "end", "val", "accn", "fy", "fp", "form", "filed")}})
    df = pd.DataFrame(rows)
    out = []
    for metric, concepts in KEY_CONCEPTS.items():
        sel = df[df["concept"].isin(concepts) & df["form"].isin(ANNUAL_FORMS) & df["start"].notna()].copy()
        if sel.empty:
            continue
        sel["days"] = (pd.to_datetime(sel["end"]) - pd.to_datetime(sel["start"])).dt.days
        sel = sel[sel["days"] > 300].sort_values("filed").groupby("end", as_index=False).first()
        sel.insert(0, "metric", metric)
        sel["ticker"], sel["cik"] = ticker, cik
        out.append(sel[["metric", "concept", "start", "end", "val", "unit", "accn", "ticker", "cik"]])
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()

for ticker in FILERS:
    out_path = XBRL_OUT / f"{ticker}_key_metrics.parquet"
    if out_path.exists():
        print(f"{ticker}: metrics cached")
        continue
    m = curate_metrics(ticker)
    m.to_parquet(out_path, index=False)
    print(f"{ticker}: {len(m)} metric-periods ({', '.join(sorted(m['metric'].unique())) if len(m) else 'none'})")

NVDA: metrics cached
AMD: metrics cached
INTC: metrics cached
AVGO: metrics cached
QCOM: metrics cached
MU: metrics cached
AAPL: metrics cached
MSFT: metrics cached
AMZN: metrics cached
GOOGL: metrics cached
META: metrics cached
TSM: metrics cached
ASML: metrics cached


## Stage 3 — Parse + chunk every filing

Same part-aware segmentation (notebook 03) and table-aware chunker (notebook 04). Section scope per form:
10-K → I.1, I.1A, II.7 · 10-Q → I.2, II.1A · **20-F → I.3, I.4, I.5** (Risk/Business/Operating Review;
20-F item numbers verified against live TSMC + ASML filings).

In [4]:
import sec_parser as sp

ITEM_RE = re.compile(r"^item\s+(\d+[a-z]?)[\.\:\s]", re.IGNORECASE)  # \s covers 20-F thin-space  
PART_RE = re.compile(r"^part\s+(i{1,3}|iv)\b", re.IGNORECASE)
NOISE = {"IrrelevantElement", "PageHeaderElement", "PageNumberElement", "EmptyElement",
         "NotYetClassifiedElement", "ImageElement", "IntroductorySectionElement"}
HEADINGS = {"TitleElement", "TopSectionTitle"}
KEEP = {"10-K": ["I.1", "I.1A", "II.7"], "10-Q": ["I.2", "II.1A"], "20-F": ["I.3", "I.4", "I.5"]}
RISK_SECTIONS = {"10-K": "I.1A", "10-Q": "II.1A", "20-F": "I.3"}
SEP = "\n\n"
TARGET_TOKENS, MAX_TOKENS = 700, 1100
SENTENCE_RE = re.compile(r"(?<=[.!?])\s+")

# --- Fallback for custom-layout filings with NO inline "Item N." headings (Intel's integrated 10-K):
# section names ride on running page-headers like "Risk Factors44" (name + page number).
PAGEHEAD_RE = re.compile(r"^([A-Za-z][A-Za-z &,’'\-]{2,60}?)\s*(\d{1,3})$")
PAGEHEAD_MAP = {
    "10-K": {"our business": "I.1", "our strategy": "I.1", "overview and our strategy": "I.1",
             "fundamentals of our business": "I.1",
             "risk factors": "I.1A", "risk factors and other key information": "I.1A",
             "md&a": "II.7"},
    "10-Q": {"md&a": "I.2", "risk factors": "II.1A"},
    "20-F": {"risk factors": "I.3"},  # ASML integrated report: only its risk section is cleanly marked
}

def _norm_marker(text: str) -> str:
    """Normalize candidate section markers: strip trailing page digits (Intel 'MD&A44')
    and '(continued)' suffixes (ASML 'Risk factors (continued)')."""
    t = re.sub(r"\s*\(continued\)\s*$", "", text.strip(), flags=re.I)
    t = re.sub(r"\s*\d{1,3}$", "", t).strip()
    return t.lower()

def n_tokens(text: str) -> int:
    return len(ENC.encode(text, disallowed_special=()))

def _segment_by_items(elements) -> list[dict]:
    part, sid, stitle, rows = "I", None, None, []
    for idx, el in enumerate(elements):
        et, text = type(el).__name__, (el.text or "").strip()
        if not text or et in NOISE:
            continue
        if et in HEADINGS and len(text) < 200:
            if (pm := PART_RE.match(text)):
                part = pm.group(1).upper(); continue
            if (m := ITEM_RE.match(text)):
                sid, stitle = f"{part}.{m.group(1).upper()}", text; continue
        if sid is None:
            continue
        rows.append({"element_index": idx, "section_id": sid, "section_title": stitle,
                     "element_type": et, "text": text})
    return rows

def _segment_by_pageheaders(elements, form: str) -> list[dict]:
    """Custom-layout fallback. Section markers come in two styles:
    - Intel: running headers 'Risk Factors44' (name + page number) on any element type
    - ASML: 'Risk factors' heading + 'Risk factors (continued)' PageHeaderElements per page
    A PageHeaderElement naming a DIFFERENT section ends the current one (page boundary)."""
    mapping = PAGEHEAD_MAP.get(form, {})
    sid, cur_name, rows = None, None, []
    for idx, el in enumerate(elements):
        et, text = type(el).__name__, (el.text or "").strip()
        if not text:
            continue
        if len(text) < 80:
            norm = _norm_marker(text)
            is_numbered = bool(PAGEHEAD_RE.match(text))
            if norm in mapping and (is_numbered or et == "PageHeaderElement" or et in HEADINGS):
                sid, cur_name = mapping[norm], norm
                continue
            if is_numbered:                 # Intel-style header for an UNMAPPED section
                sid, cur_name = None, None
                continue
            if et == "PageHeaderElement" and sid is not None and norm != cur_name:
                sid, cur_name = None, None  # ASML-style: page belongs to another section
                continue
        if sid is None or et in NOISE:
            continue
        rows.append({"element_index": idx, "section_id": sid, "section_title": cur_name,
                     "element_type": et, "text": text})
    return rows

def segment(html: str, form: str) -> pd.DataFrame:
    elements = sp.Edgar10QParser().parse(html)
    rows = _segment_by_items(elements)
    keep = set(KEEP.get(form, []))
    # Fallback when the item path finds NOTHING USABLE — covers Intel (zero item headings)
    # and ASML (only a junk cover-page 'Item 17 / Item 18' checkbox heading).
    if not any(r["section_id"] in keep for r in rows):
        rows = _segment_by_pageheaders(elements, form)
    return pd.DataFrame(rows, columns=["element_index", "section_id", "section_title", "element_type", "text"])

def split_oversized(text, abs_start):
    pieces, buf, buf_start, cursor = [], [], abs_start, abs_start
    for sent in SENTENCE_RE.split(text):
        if buf and n_tokens(" ".join(buf + [sent])) > MAX_TOKENS:
            joined = " ".join(buf); pieces.append((joined, buf_start, buf_start + len(joined)))
            buf, buf_start = [sent], cursor
        else:
            buf.append(sent)
        cursor += len(sent) + 1
    if buf:
        joined = " ".join(buf)
        pieces.append((joined, buf_start, min(buf_start + len(joined), abs_start + len(text))))
    return pieces

def chunk_filing(meta: dict, elements_df: pd.DataFrame) -> tuple[list[dict], list[dict]]:
    """Return (section_text_rows, chunk_rows) for one filing, restricted to KEEP sections."""
    keep = KEEP.get(meta["form"], [])
    st_rows, chunk_rows = [], []
    if elements_df.empty:  # nothing segmentable — caller reports it; never crash the batch
        return st_rows, chunk_rows
    for sid, grp in elements_df[elements_df["section_id"].isin(keep)].groupby("section_id", sort=False):
        grp = grp.sort_values("element_index")
        offsets, cursor, parts = [], 0, []
        for _, row in grp.iterrows():
            offsets.append((row, cursor, cursor + len(row["text"])))
            parts.append(row["text"]); cursor += len(row["text"]) + len(SEP)
        full_text = SEP.join(parts)
        st_rows.append({"accession_no": meta["accession_no"], "section_id": sid, "form": meta["form"],
                        "filing_date": meta["filing_date"], "section_title": grp.iloc[0]["section_title"],
                        "text": full_text, "n_chars": len(full_text)})
        buf, sub_heading, seq = [], None, 0
        def flush(kind):
            nonlocal seq
            if not buf: return
            start, end = buf[0][1], buf[-1][2]
            text = full_text[start:end]
            chunk_rows.append({"chunk_id": f"{meta['accession_no']}:{sid}:{seq:04d}",
                               "ticker": meta["ticker"], "cik": meta["cik"], "form": meta["form"],
                               "filing_date": meta["filing_date"], "accession_no": meta["accession_no"],
                               "section_id": sid, "section_title": grp.iloc[0]["section_title"],
                               "sub_heading": sub_heading, "kind": kind, "text": text,
                               "char_start": start, "char_end": end, "n_tokens": n_tokens(text),
                               "source_url": meta["source_url"]})
            seq += 1; buf.clear()
        for row, start, end in offsets:
            if row["element_type"] == "TitleElement":
                flush("prose"); sub_heading = row["text"][:150]; continue
            if row["element_type"] == "TableElement":
                flush("prose"); buf.append((row, start, end)); flush("table"); continue
            if n_tokens(row["text"]) > MAX_TOKENS:
                flush("prose")
                for piece, ps, pe in split_oversized(row["text"], start):
                    buf.append((row, ps, pe)); flush("prose")
                continue
            buf.append((row, start, end))
            if n_tokens(full_text[buf[0][1]:buf[-1][2]]) >= TARGET_TOKENS:
                flush("prose")
        flush("prose")
    return st_rows, chunk_rows

for ticker in FILERS:
    chunks_path = CHUNKS_DIR / f"{ticker}_chunks.parquet"
    if ticker == "NVDA":
        chunks_path = CHUNKS_DIR / "nvda_chunks.parquet"  # notebook 04's output — reused as-is
    if chunks_path.exists() and len(pd.read_parquet(chunks_path)):
        print(f"{ticker}: chunks cached ({chunks_path.name})")
        continue
    st_all, ch_all = [], []
    for meta in manifest[ticker]:
        html = (PROJECT_ROOT / meta["local_path"]).read_text(encoding="utf-8")
        st, ch = chunk_filing(meta, segment(html, meta["form"]))
        if not ch:
            print(f"  WARNING {ticker} {meta['form']} {meta['filing_date']}: no keep-sections segmented — review layout")
        st_all.extend(st); ch_all.extend(ch)
    pd.DataFrame(st_all).to_parquet(SECTION_TEXT_DIR / f"{ticker}_section_texts.parquet", index=False)
    pd.DataFrame(ch_all).to_parquet(chunks_path, index=False)
    print(f"{ticker}: {len(ch_all)} chunks / {len(st_all)} sections / {sum(c['n_tokens'] for c in ch_all):,} tokens")

chunk_files = {t: (CHUNKS_DIR / ("nvda_chunks.parquet" if t == "NVDA" else f"{t}_chunks.parquet")) for t in FILERS}
all_chunks = pd.concat([pd.read_parquet(p) for p in chunk_files.values()], ignore_index=True)
print(f"universe: {len(all_chunks):,} chunks, {all_chunks['n_tokens'].sum():,} tokens")

NVDA: chunks cached (nvda_chunks.parquet)
AMD: chunks cached (AMD_chunks.parquet)
INTC: chunks cached (INTC_chunks.parquet)
AVGO: chunks cached (AVGO_chunks.parquet)
QCOM: chunks cached (QCOM_chunks.parquet)
MU: chunks cached (MU_chunks.parquet)
AAPL: chunks cached (AAPL_chunks.parquet)
MSFT: chunks cached (MSFT_chunks.parquet)
AMZN: chunks cached (AMZN_chunks.parquet)
GOOGL: chunks cached (GOOGL_chunks.parquet)
META: chunks cached (META_chunks.parquet)
TSM: chunks cached (TSM_chunks.parquet)
ASML: chunks cached (ASML_chunks.parquet)
universe: 4,933 chunks, 1,716,535 tokens


## Stage 4 — Deterministic graph load (Filings, Sections, Metrics — no LLM)

In [5]:
with driver.session() as session:
    session.run("""UNWIND $rows AS row
        MATCH (c:Company {ticker: row.ticker})
        MERGE (f:Filing {accession_no: row.accession_no})
        SET f.form = row.form, f.filing_date = date(row.filing_date), f.url = row.source_url
        MERGE (c)-[:FILED {date: date(row.filing_date)}]->(f)""", rows=ALL_FILINGS)
    section_rows = []
    for ticker in FILERS:
        st_path = SECTION_TEXT_DIR / ("nvda_section_texts.parquet" if ticker == "NVDA" else f"{ticker}_section_texts.parquet")
        st = pd.read_parquet(st_path)
        section_rows += [{"section_key": f"{r.accession_no}:{r.section_id}", "accession_no": r.accession_no,
                          "section_id": r.section_id, "title": r.section_title, "n_chars": int(r.n_chars)}
                         for r in st.itertuples()]
    session.run("""UNWIND $rows AS row
        MATCH (f:Filing {accession_no: row.accession_no})
        MERGE (s:FilingSection {section_key: row.section_key})
        SET s.section_id = row.section_id, s.title = row.title, s.n_chars = row.n_chars
        MERGE (f)-[:HAS_SECTION]->(s)""", rows=section_rows)
    metric_rows = []
    for ticker in FILERS:
        mp = XBRL_OUT / f"{ticker}_key_metrics.parquet"
        if not mp.exists() and ticker == "NVDA":
            mp = XBRL_OUT / "NVDA_key_metrics.parquet"
        if not mp.exists():
            continue
        for r in pd.read_parquet(mp).itertuples():
            metric_rows.append({"metric_id": f"{int(r.cik)}:{r.metric}:{r.end}", "cik": int(r.cik),
                                "metric": r.metric, "concept": r.concept, "value": float(r.val),
                                "unit": r.unit, "period_start": r.start, "period_end": r.end, "accn": r.accn})
    session.run("""UNWIND $rows AS row
        MATCH (c:Company {cik: row.cik})
        MERGE (m:Metric {metric_id: row.metric_id})
        SET m.metric = row.metric, m.concept = row.concept, m.value = row.value, m.unit = row.unit,
            m.period_start = date(row.period_start), m.period_end = date(row.period_end)
        MERGE (c)-[rel:REPORTS_METRIC]->(m) SET rel.accession_no = row.accn""", rows=metric_rows)
    stats = session.run("MATCH (f:Filing) RETURN count(f) AS f").single()
print(f"deterministic layer: {len(ALL_FILINGS)} filings in manifest, {stats['f']} Filing nodes, "
      f"{len(section_rows)} sections, {len(metric_rows)} metrics")

deterministic layer: 59 filings in manifest, 59 Filing nodes, 151 sections, 731 metrics


## Stage 5 — LLM extraction (the paid stage). Scope + honest cost estimate BEFORE any spend.

In [6]:
HIST_ANNUALS = 1  # how many PRIOR annuals contribute risk-only chunks. 1 = latest + one prior
                  # = two time points per company for notebook 13's lineages. Raise if budget allows.

def extraction_scope(ticker: str) -> pd.DataFrame:
    """Latest annual (all kept sections) + latest quarterly + HIST_ANNUALS prior annuals (risk-only)."""
    ch = pd.read_parquet(chunk_files[ticker])
    if ch.empty or "form" not in ch.columns:  # filer with no segmentable chunks — skip gracefully
        return pd.DataFrame(columns=["chunk_id", "ticker", "form", "accession_no", "section_id",
                                     "filing_date", "n_tokens", "text", "section_title", "sub_heading"])
    name, annual_form, quarterly_form = FILERS[ticker]
    annuals = sorted(ch[ch["form"] == annual_form]["accession_no"].unique(),
                     key=lambda a: ch[ch["accession_no"] == a]["filing_date"].iloc[0])
    parts = []
    if annuals:
        parts.append(ch[ch["accession_no"] == annuals[-1]])                      # latest annual: everything
        risk = RISK_SECTIONS[annual_form]
        hist = annuals[-(1 + HIST_ANNUALS):-1]
        parts.append(ch[ch["accession_no"].isin(hist) & (ch["section_id"] == risk)])  # history: risks only
    if quarterly_form:
        qs = ch[ch["form"] == quarterly_form]
        if len(qs):
            parts.append(qs[qs["accession_no"] == qs["accession_no"].max()])
    return pd.concat(parts, ignore_index=True).drop_duplicates("chunk_id") if parts else ch.iloc[0:0]

done_ids = set()
for jl in EXTRACT_DIR.glob("*_extractions.jsonl"):
    done_ids |= {json.loads(l)["chunk_id"] for l in jl.open(encoding="utf-8") if l.strip()}

scopes = {t: extraction_scope(t) for t in FILERS}
todo = {t: s[~s["chunk_id"].isin(done_ids)] for t, s in scopes.items()}

# --- Honest cost estimate. Per extractor call = instruction/schema overhead + chunk; the critic
# (Haiku 4.5, $1/$5 per Mtok) only runs on the ~25% of chunks whose relations survive the quote gate;
# average output measured from the NVDA PoC (~300 tokens, mostly small/empty JSON).
n_todo = sum(len(s) for s in todo.values())
chunk_tokens = int(sum(s["n_tokens"].sum() for s in todo.values()))
OVERHEAD, CRITIC_FRACTION, OUT_PER_CHUNK = 800, 0.25, 300
extract_in = (n_todo * OVERHEAD + chunk_tokens) / 1e6 * 3
extract_out = n_todo * OUT_PER_CHUNK / 1e6 * 15
critic_cost = CRITIC_FRACTION * ((n_todo * 500 + chunk_tokens) / 1e6 * 1 + n_todo * 40 / 1e6 * 5)
likely = extract_in + extract_out + critic_cost
print(pd.DataFrame({"chunks_in_scope": {t: len(s) for t, s in scopes.items()},
                    "already_done": {t: len(scopes[t]) - len(todo[t]) for t in FILERS},
                    "to_extract": {t: len(s) for t, s in todo.items()}}).to_string())
print(f"\nremaining: {n_todo} chunks, {chunk_tokens:,} chunk tokens")
print(f"ESTIMATED COST ~${likely:.2f} likely / ~${likely * 1.5:.2f} worst case")
print(f"  = Sonnet extractor in ${extract_in:.2f} + out ${extract_out:.2f} + Haiku critic ${critic_cost:.2f}")
print("Checkpointed per chunk — interruptions never re-bill. Re-run this cell to refresh after any stop.")

       chunks_in_scope  already_done  to_extract
NVDA               165           165           0
AMD                207           207           0
INTC               167           167           0
AVGO               159           159           0
QCOM               207           207           0
MU                 207           207           0
AAPL               139           139           0
MSFT               139           139           0
AMZN               163           163           0
GOOGL              213           213           0
META               304            25         279
TSM                 90             0          90
ASML                58             0          58

remaining: 427 chunks, 191,884 chunk tokens
ESTIMATED COST ~$3.64 likely / ~$5.47 worst case
  = Sonnet extractor in $1.60 + out $1.92 + Haiku critic $0.12
Checkpointed per chunk — interruptions never re-bill. Re-run this cell to refresh after any stop.


In [7]:
import time as _time
import litellm
from pydantic import BaseModel, Field, ValidationError
from litellm import completion

LLM_MODEL = os.environ["LLM_MODEL"]
CRITIC_MODEL = "anthropic/claude-haiku-4-5"  # binary support-check — Haiku is 3x cheaper and verified live
RELATION_TYPES = ["SUPPLIES_TO", "DEPENDS_ON", "CUSTOMER_OF", "COMPETES_WITH"]

class Relation(BaseModel):
    source_entity: str; relation: str; target_entity: str
    evidence_quote: str = Field(description="VERBATIM quote (<=40 words)")
class RiskFactor(BaseModel):
    summary: str; category: str
    evidence_quote: str = Field(description="VERBATIM quote (<=40 words)")
class Product(BaseModel):
    name: str; type: str
class ChunkExtraction(BaseModel):
    relations: list[Relation] = []; risk_factors: list[RiskFactor] = []; products: list[Product] = []
class CriticVerdict(BaseModel):
    verdicts: list[bool]

EXTRACTOR_PROMPT = (PROJECT_ROOT / "artifacts/prompts/extractor.txt").read_text(encoding="utf-8")
CRITIC_PROMPT = (PROJECT_ROOT / "artifacts/prompts/critic.txt").read_text(encoding="utf-8")

# Transient failures worth waiting out (network blips like WinError 10054, 429s, 5xx). Anything
# else (auth, bad request) raises immediately — no blind retries on unknown errors.
TRANSIENT = (litellm.APIConnectionError, litellm.ServiceUnavailableError,
             litellm.InternalServerError, litellm.RateLimitError, litellm.Timeout)

def llm_json(prompt: str, model_cls, model: str = None, max_tokens: int = 4000, use_thinking_off: bool = True):
    """Structured LLM call, hardened against every failure mode hit so far (each verified live):
    - no sampling params; thinking disabled on Sonnet (omit for Haiku — thinking-off by default)
    - finish_reason=='length' -> regenerate fresh with doubled budget (never 'fix' truncated JSON)
    - empty content -> fresh retry
    - schema mismatch -> one correction turn
    - transient network/API errors -> exponential backoff (10s/30s/90s), in-flight retries via num_retries
    """
    model = model or LLM_MODEL
    kwargs = {"thinking": {"type": "disabled"}} if use_thinking_off else {}
    messages, budget, last_err = [{"role": "user", "content": prompt}], max_tokens, "unknown"
    for attempt in range(4):
        try:
            resp = completion(model=model, messages=messages, max_tokens=budget, num_retries=2, **kwargs)
        except TRANSIENT as e:
            wait = [15, 60, 180, 300][attempt]  # long tail: 529 overload incidents can last minutes
            print(f"    transient error ({type(e).__name__}) — waiting {wait}s then retrying")
            _time.sleep(wait)
            last_err = f"transient: {type(e).__name__}"
            continue
        choice = resp.choices[0]
        content = choice.message.content
        if not content:
            last_err = f"empty response ({choice.finish_reason})"
            messages = [{"role": "user", "content": prompt}]; continue
        if choice.finish_reason == "length":
            budget = min(budget * 2, 8000); last_err = "truncated"
            messages = [{"role": "user", "content": prompt}]; continue
        raw = re.sub(r"^```(json)?|```$", "", content.strip(), flags=re.MULTILINE).strip()
        try:
            return model_cls.model_validate_json(raw)
        except ValidationError as e:
            last_err = str(e)[:300]
            messages = [{"role": "user", "content": prompt}, {"role": "assistant", "content": raw},
                        {"role": "user", "content": f"Invalid JSON for the schema: {e}. Reply with corrected JSON only."}]
    raise RuntimeError(f"llm_json failed after 4 attempts — {last_err}")

def normalize(s: str) -> str:
    return re.sub(r"[\s’'\"“”]+", " ", s).strip().lower()

def quote_in_chunk(quote: str, chunk_text: str) -> bool:
    return normalize(quote) in normalize(chunk_text)

schema_json = json.dumps(ChunkExtraction.model_json_schema(), indent=None)

for ticker in FILERS:
    batch = todo[ticker]
    if batch.empty:
        continue
    name = FILERS[ticker][0]
    out_path = EXTRACT_DIR / f"{ticker.lower()}_extractions.jsonl"
    if ticker == "NVDA":
        out_path = EXTRACT_DIR / "nvda_extractions.jsonl"  # append to the PoC file
    print(f"--- {ticker}: {len(batch)} chunks ---")
    with out_path.open("a", encoding="utf-8") as sink:
        for n, (_, c) in enumerate(batch.iterrows(), 1):
            prompt = EXTRACTOR_PROMPT.format(
                ticker=c["ticker"], ticker_name=name, form=c["form"], filing_date=c["filing_date"],
                section_title=c["section_title"],
                sub_heading=f", sub-heading \"{c['sub_heading']}\"" if c["sub_heading"] else "",
                schema=schema_json, chunk_text=c["text"])
            extraction = llm_json(prompt, ChunkExtraction)
            relations = [r for r in extraction.relations if quote_in_chunk(r.evidence_quote, c["text"])]
            risks = [r for r in extraction.risk_factors if quote_in_chunk(r.evidence_quote, c["text"])]
            if relations:
                claims = "\n".join(f"{j+1}. {r.source_entity} {r.relation} {r.target_entity}"
                                   for j, r in enumerate(relations))
                verdict = llm_json(CRITIC_PROMPT.format(chunk_text=c["text"], claims=claims), CriticVerdict,
                                   model=CRITIC_MODEL, max_tokens=600, use_thinking_off=False)
                kept = [r for r, ok in zip(relations, verdict.verdicts) if ok] \
                    if len(verdict.verdicts) == len(relations) else relations
            else:
                kept = []
            sink.write(json.dumps({"chunk_id": c["chunk_id"], "ticker": ticker,
                                   "accession_no": c["accession_no"], "section_id": c["section_id"],
                                   "relations": [r.model_dump() for r in kept],
                                   "risk_factors": [r.model_dump() for r in risks],
                                   "products": [p.model_dump() for p in extraction.products]}) + "\n")
            sink.flush()
            if n % 25 == 0:
                print(f"  {n}/{len(batch)}")
print("extraction complete")

--- META: 279 chunks ---
  25/279
  50/279
  75/279
  100/279
  125/279
  150/279
  175/279
  200/279
  225/279
  250/279
  275/279
--- TSM: 90 chunks ---
  25/90
  50/90
  75/90
--- ASML: 58 chunks ---
  25/58
  50/58
extraction complete


## Stage 6 — Resolve entities, embed new chunks, load knowledge (filer-aware)

In [8]:
import hashlib
from difflib import SequenceMatcher
from sentence_transformers import SentenceTransformer

LEGAL_SUFFIXES = re.compile(r"\b(incorporated|corporation|corp|inc|ltd|limited|llc|plc|co|company|holdings?|nv|sa|ag|kk)\b\.?", re.I)
def normalize_name(s):
    s = re.sub(r"[^\w\s]", " ", s.lower()); s = LEGAL_SUFFIXES.sub(" ", s)
    return re.sub(r"\s+", " ", s).strip()
ALIAS_LOOKUP = {normalize_name(a): n for n, spec in CANONICAL.items() for a in [n] + spec["aliases"]}
def resolve(raw, threshold=0.90):
    norm = normalize_name(raw)
    if not norm: return None
    if norm in ALIAS_LOOKUP: return ALIAS_LOOKUP[norm]
    best, score = None, 0.0
    for alias, name in ALIAS_LOOKUP.items():
        s = SequenceMatcher(None, norm, alias).ratio()
        if s > score: best, score = name, s
    return best if score >= threshold else None

def load_embedder(model_name):
    try: return SentenceTransformer(model_name, local_files_only=True)
    except OSError: return SentenceTransformer(model_name)
emb_model = load_embedder(os.getenv("EMBEDDING_MODEL", "Qwen/Qwen3-Embedding-0.6B"))

alias_patterns = [(re.compile(rf"\b{re.escape(a)}\b", re.I), spec["entity_id"])
                  for nm, spec in CANONICAL.items() for a in {nm, *spec["aliases"]}]
def mentioned(text):
    return sorted({eid for pat, eid in alias_patterns if pat.search(text)})

with driver.session() as s:
    have_spans = {r["id"] for r in s.run("MATCH (e:EvidenceSpan) RETURN e.chunk_id AS id")}

dropped_all = []
for ticker in FILERS:
    ch = pd.read_parquet(chunk_files[ticker])
    scope_ids = set(scopes[ticker]["chunk_id"])
    new = ch[ch["chunk_id"].isin(scope_ids - have_spans)]
    if len(new):  # embed + load EvidenceSpans for chunks not yet in the graph
        cache = EMB_DIR / f"{ticker}_chunk_embeddings.parquet"
        if cache.exists() and set(pd.read_parquet(cache)["chunk_id"]) >= set(new["chunk_id"]):
            cdf = pd.read_parquet(cache).set_index("chunk_id")
            vecs = np.vstack(cdf.loc[new["chunk_id"], "embedding"].to_numpy())
        else:
            inputs = (new["sub_heading"].fillna("") + "\n" + new["text"]).str.strip().tolist()
            vecs = emb_model.encode(inputs, batch_size=8, show_progress_bar=True, normalize_embeddings=True)
            pd.DataFrame({"chunk_id": new["chunk_id"], "embedding": list(vecs)}).to_parquet(cache, index=False)
        rows = [{"chunk_id": r.chunk_id, "text": r.text, "kind": r.kind,
                 "section_key": f"{r.accession_no}:{r.section_id}", "sub_heading": r.sub_heading,
                 "char_start": int(r.char_start), "char_end": int(r.char_end), "n_tokens": int(r.n_tokens),
                 "source_url": r.source_url, "embedding": v.tolist(), "mentions": mentioned(r.text)}
                for r, v in zip(new.itertuples(), vecs)]
        with driver.session() as s:
            for i in range(0, len(rows), 100):
                s.run("""UNWIND $rows AS row
                    MERGE (e:EvidenceSpan {chunk_id: row.chunk_id})
                    SET e.text = row.text, e.kind = row.kind, e.sub_heading = row.sub_heading,
                        e.char_start = row.char_start, e.char_end = row.char_end,
                        e.n_tokens = row.n_tokens, e.source_url = row.source_url, e.embedding = row.embedding
                    WITH e, row MATCH (sec:FilingSection {section_key: row.section_key})
                    MERGE (e)-[:FROM_SECTION]->(sec)
                    WITH e, row UNWIND row.mentions AS eid
                    MATCH (c:Company {cik: eid}) MERGE (e)-[:MENTIONS]->(c)""", rows=rows[i:i+100])
        print(f"{ticker}: {len(rows)} EvidenceSpans loaded")

    # knowledge load from this filer's extraction jsonl
    jl = EXTRACT_DIR / ("nvda_extractions.jsonl" if ticker == "NVDA" else f"{ticker.lower()}_extractions.jsonl")
    if not jl.exists():
        continue
    recs = [json.loads(l) for l in jl.open(encoding="utf-8") if l.strip()]
    meta_by_accn = {m["accession_no"]: m for m in manifest.get(ticker, [])}
    ch_meta = ch.set_index("chunk_id")["filing_date"].to_dict()
    rel_rows, risk_rows, prod_rows = [], [], []
    for rec in recs:
        fdate = ch_meta.get(rec["chunk_id"])
        if fdate is None: continue
        for rel in rec["relations"]:
            src, tgt = resolve(rel["source_entity"]), resolve(rel["target_entity"])
            if src and tgt and src != tgt:
                rel_rows.append({"src": CANONICAL[src]["entity_id"], "tgt": CANONICAL[tgt]["entity_id"],
                                 "type": rel["relation"], "chunk_id": rec["chunk_id"],
                                 "quote": rel["evidence_quote"], "date": fdate})
            else:
                dropped_all.append({"ticker": ticker, **rel})
        for risk in rec["risk_factors"]:
            rid = hashlib.sha1(f"{rec['chunk_id']}|{risk['summary']}".encode()).hexdigest()[:16]
            risk_rows.append({"risk_id": rid, "summary": risk["summary"], "category": risk["category"],
                              "chunk_id": rec["chunk_id"], "quote": risk["evidence_quote"],
                              "date": fdate, "filer_cik": int(ch["cik"].iloc[0])})
        for p in rec["products"]:
            prod_rows.append({"name": p["name"], "type": p["type"], "chunk_id": rec["chunk_id"]})
    with driver.session() as s:
        have_risks = {r["id"] for r in s.run("MATCH (rf:RiskFactor) RETURN rf.risk_id AS id")}
    new_risks = [r for r in risk_rows if r["risk_id"] not in have_risks]
    if new_risks:
        r_embs = emb_model.encode([r["summary"] for r in new_risks], normalize_embeddings=True)
        for r, v in zip(new_risks, r_embs): r["embedding"] = v.tolist()
    with driver.session() as s:
        for rt in RELATION_TYPES:
            b = [r for r in rel_rows if r["type"] == rt]
            if b:
                s.run(f"""UNWIND $rows AS row
                    MATCH (a:Company {{cik: row.src}}), (t:Company {{cik: row.tgt}})
                    MERGE (a)-[r:{rt}]->(t)
                    ON CREATE SET r.start_date = date(row.date), r.status = 'Active',
                                  r.evidence_chunk_ids = [row.chunk_id], r.evidence_quote = row.quote
                    ON MATCH SET r.evidence_chunk_ids = CASE WHEN row.chunk_id IN r.evidence_chunk_ids
                                  THEN r.evidence_chunk_ids ELSE r.evidence_chunk_ids + row.chunk_id END""", rows=b)
        if new_risks:
            s.run("""UNWIND $rows AS row
                MERGE (rf:RiskFactor {risk_id: row.risk_id})
                SET rf.summary = row.summary, rf.category = row.category, rf.embedding = row.embedding
                WITH rf, row MATCH (e:EvidenceSpan {chunk_id: row.chunk_id})
                MERGE (rf)-[:HAS_EVIDENCE {quote: row.quote}]->(e)
                WITH rf, row MATCH (filer:Company {cik: row.filer_cik})
                MERGE (filer)-[d:DISCLOSES_RISK]->(rf)
                ON CREATE SET d.start_date = date(row.date), d.status = 'Active'""", rows=new_risks)
        if prod_rows:
            s.run("""UNWIND $rows AS row
                MERGE (p:Product {name: row.name}) SET p.type = coalesce(p.type, row.type)
                WITH p, row MATCH (e:EvidenceSpan {chunk_id: row.chunk_id})
                MERGE (p)-[:MENTIONED_IN]->(e)""", rows=prod_rows)
    print(f"{ticker}: +{len(rel_rows)} relation instances, +{len(new_risks)} risks, {len(prod_rows)} product mentions")

pd.DataFrame(dropped_all).to_parquet(EXTRACT_DIR / "resolution_report_universe.parquet", index=False)
print(f"unresolved entities logged: {len(dropped_all)} (dictionary growth candidates)")

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

NVDA: +17 relation instances, +597 risks, 250 product mentions


Batches:   0%|          | 0/26 [00:00<?, ?it/s]

AMD: 207 EvidenceSpans loaded
AMD: +47 relation instances, +871 risks, 373 product mentions


Batches:   0%|          | 0/21 [00:00<?, ?it/s]

...

META: 304 EvidenceSpans loaded
META: +26 relation instances, +1072 risks, 236 product mentions


Batches:   0%|          | 0/12 [00:00<?, ?it/s]

TSM: 90 EvidenceSpans loaded
TSM: +0 relation instances, +217 risks, 117 product mentions


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

ASML: 58 EvidenceSpans loaded
ASML: +0 relation instances, +333 risks, 25 product mentions
unresolved entities logged: 372 (dictionary growth candidates)


## Stage 7 — Export controls: Federal Register → `ExportControl` nodes + `AFFECTED_BY` edges

BIS rules (2022+) become `ExportControl` nodes. **Linking heuristic (documented, refined in M6):** a company
is `AFFECTED_BY` a rule when it discloses an Export Controls-category risk whose evidence text matches the
rule's topic keywords; evidence chunk ids ride on the edge.

In [14]:
import urllib.parse
import urllib.error
import time as _t

fr_params = {
    "conditions[agencies][]": "industry-and-security-bureau", "conditions[type][]": "RULE",
    "conditions[term]": 'semiconductor OR "advanced computing" OR "export controls"',
    "conditions[publication_date][gte]": "2022-01-01", "per_page": "50", "order": "newest",
    "fields[]": ["document_number", "title", "publication_date", "html_url", "abstract"],
}
fr_url = "https://www.federalregister.gov/api/v1/documents.json?" + urllib.parse.urlencode(fr_params, doseq=True)
fr_cache = PROJECT_ROOT / "data/raw/federal_register_bis_rules.json"
if not fr_cache.exists():
    for attempt in range(4):  # free API, but DNS/network can blip — retry with backoff
        try:
            req = urllib.request.Request(fr_url, headers={"User-Agent": SEC_USER_AGENT})
            fr_cache.write_bytes(urllib.request.urlopen(req, timeout=30).read())
            break
        except (urllib.error.URLError, OSError) as e:
            if attempt == 3:
                raise RuntimeError(f"Federal Register API unreachable after 4 tries — check internet/DNS: {e}")
            wait = [5, 15, 45][attempt]
            print(f"network error ({e}) — retrying in {wait}s")
            _t.sleep(wait)
rules = json.loads(fr_cache.read_text())["results"]

TOPIC_KEYWORDS = {  # rule-title keyword -> evidence-text keywords
    "entity list": ["entity list"],
    "advanced computing": ["advanced computing", "ai chip", "accelerator"],
    "semiconductor manufacturing": ["manufacturing equipment", "semiconductor manufacturing"],
    "artificial intelligence": ["artificial intelligence", "ai diffusion"],
}

with driver.session() as s:
    s.run("""UNWIND $rows AS row
        MERGE (x:ExportControl {rule_id: row.document_number})
        SET x.title = row.title, x.date = date(row.publication_date), x.url = row.html_url,
            x.abstract = left(coalesce(row.abstract, ''), 1000)""", rows=rules)
    # companies with Export Controls-category risks, plus their evidence text
    exposures = s.run("""MATCH (c:Company)-[:DISCLOSES_RISK]->(rf:RiskFactor {category: 'Export Controls'})
                              -[:HAS_EVIDENCE]->(e:EvidenceSpan)
                         RETURN c.cik AS cik, c.name AS name, collect(DISTINCT e.chunk_id) AS chunks,
                                left(reduce(t = '', x IN collect(e.text)[..5] | t + ' ' + x), 8000) AS text""").data()
    edges = []
    for exp in exposures:
        text_l = exp["text"].lower()
        for rule in rules:
            title_l = rule["title"].lower()
            for topic, ev_keywords in TOPIC_KEYWORDS.items():
                if topic in title_l and any(k in text_l for k in ev_keywords):
                    edges.append({"cik": exp["cik"], "rule_id": rule["document_number"],
                                  "chunks": exp["chunks"][:10], "date": rule["publication_date"]})
                    break
    s.run("""UNWIND $rows AS row
        MATCH (c:Company {cik: row.cik}), (x:ExportControl {rule_id: row.rule_id})
        MERGE (c)-[r:AFFECTED_BY]->(x)
        ON CREATE SET r.start_date = date(row.date), r.status = 'Active', r.evidence_chunk_ids = row.chunks""",
          rows=edges)
    n_x = s.run("MATCH (:ExportControl) RETURN count(*) AS n").single()["n"]
    n_ab = s.run("MATCH ()-[r:AFFECTED_BY]->() RETURN count(r) AS n").single()["n"]
print(f"{n_x} ExportControl nodes, {n_ab} AFFECTED_BY edges "
      f"({len(exposures)} companies disclose export-control risks)")

13 ExportControl nodes, 21 AFFECTED_BY edges (4 companies disclose export-control risks)


## M5 exit — the feasibility studies' flagship **Query D**: Meta → ... → export controls

In [15]:
QUERY_D = """
MATCH (meta:Company {ticker: 'META'}), (x:ExportControl)
MATCH p = shortestPath((meta)-[:SUPPLIES_TO|DEPENDS_ON|CUSTOMER_OF|COMPETES_WITH|MENTIONS|AFFECTED_BY*..6]-(x))
WITH p LIMIT 1
RETURN [n IN nodes(p) | coalesce(n.name, n.title, left(n.text, 60), n.chunk_id)] AS path_nodes,
       [r IN relationships(p) | type(r)] AS path_rels
"""
with driver.session() as s:
    row = s.run(QUERY_D).single()
if row:
    print("Query D path (Meta -> export controls):")
    for i, n in enumerate(row["path_nodes"]):
        print("   " + ("" if i == 0 else f"--[{row['path_rels'][i-1]}]--> ") + str(n)[:90])
else:
    print("No Meta->ExportControl path yet — inspect AFFECTED_BY coverage and Meta's extraction output.")

Query D path (Meta -> export controls):
   Meta
   --[CUSTOMER_OF]--> AMD
   --[AFFECTED_BY]--> Additions and Revisions to the Entity List


In [16]:
# --- M5 assertion cell ---
with driver.session() as s:
    n_filers_with_filings = s.run(
        "MATCH (c:Company)-[:FILED]->(:Filing) RETURN count(DISTINCT c) AS n").single()["n"]
    n_spans = s.run("MATCH (:EvidenceSpan) RETURN count(*) AS n").single()["n"]
    n_x = s.run("MATCH (:ExportControl) RETURN count(*) AS n").single()["n"]
    n_ab = s.run("MATCH ()-[:AFFECTED_BY]->() RETURN count(*) AS n").single()["n"]
    multi_hop = s.run("""MATCH (h:Company)-[:CUSTOMER_OF|DEPENDS_ON*1..3]->(t:Company {ticker: 'TSM'})
                         WHERE h.tier = 'Hyperscaler' OR h.ticker IN ['NVDA','AMD','AAPL']
                         RETURN count(DISTINCT h) AS n""").single()["n"]
assert n_filers_with_filings >= 13, f"expected 13 filers with filings, got {n_filers_with_filings}"
assert n_spans > 1000, f"expected >1000 EvidenceSpans at universe scale, got {n_spans}"
assert n_x >= 10 and n_ab >= 3, f"export-control layer thin: {n_x} rules, {n_ab} AFFECTED_BY"
assert multi_hop >= 1, "no multi-hop path into TSMC — inspect relation extraction"
assert row is not None, "Query D returned no path"
driver.close()
print(f"M5 COMPLETE — {n_filers_with_filings} filers, {n_spans:,} evidence spans, "
      f"{n_x} export-control rules, {n_ab} AFFECTED_BY edges, Query D path found")

M5 COMPLETE — 13 filers, 2,492 evidence spans, 13 export-control rules, 21 AFFECTED_BY edges, Query D path found
